In [ ]:
from moe_reft.olmoe.modeling_olmoe import OlmoeForCausalLM
from moe_reft.olmoe import configuration_olmoe
from moe_reft import interventions_config

model = OlmoeForCausalLM(configuration_olmoe.OlmoeInterventionsConfig(
    interventions_config=interventions_config.InterventionsConfig(
        intervention_places="after_moe",
        intervention_layers="even_only",
    ),
))

In [ ]:
from __future__ import annotations

import torch
from loguru import logger
from torch import nn
from transformers import AutoModelForCausalLM, PreTrainedModel

map_dtype=torch.bfloat16  # optional casting
map_device=torch.device("cuda")  # optional device move
        
hf_model_name_or_path="allenai/OLMoE-1B-7B-0125-Instruct"
# hf_model_name_or_path="allenai/OLMoE-1B-7B-0924-Instruct"

hf_model: PreTrainedModel = AutoModelForCausalLM.from_pretrained(
        hf_model_name_or_path,
        dtype=map_dtype if map_dtype is not None else None,
        trust_remote_code=True,
    )

src_sd = hf_model.state_dict()

In [ ]:
from moe_reft.olmoe import load_weights

intervention_patterns = [
    "*.pre_moe_intervention.*",
    "*.after_moe_intervention.*",
    "*.pre_moe_intervenetion.*",  # typo fallback
]
    # 1) Load HF model & grab its state dict
hf_model: PreTrainedModel = AutoModelForCausalLM.from_pretrained(
    hf_model_name_or_path,
    dtype=map_dtype if map_dtype is not None else None,
    trust_remote_code="True",
)
src_sd = hf_model.state_dict()

# 2) Build filtered state dict compatible with your custom model
filtered_sd, report = load_weights.build_partial_state_dict(
    src_sd=src_sd,
    dst_module=model,
    intervention_patterns=intervention_patterns,
    device=map_device,
    dtype=map_dtype,
)

# 3) Load with strict=False (so missing keys — e.g., interventions — are fine)
missing, unexpected = model.load_state_dict(filtered_sd, strict=False)

# Merge loader feedback into the report
report.skipped_missing.extend(missing)
if unexpected:
    report.skipped_missing.extend(unexpected)

# 4) Freeze everything, then unfreeze only intervention layers
for param in model.parameters():
    param.requires_grad = False


In [ ]:
for name, param in model.named_parameters():
    if load_weights._matches_any(name, intervention_patterns):
        param.requires_grad = True

# 5) Print parameter stats
total_params, trainable_params = load_weights._count_parameters(model)
print(f"Total parameters:     {total_params}")
print(f"Trainable parameters: {trainable_params}")

logger.info(f"Parameter stats — total: {total_params}, trainable: {trainable_params}")

In [ ]:
for name,param in model.named_parameters():
    if load_weights._matches_any(name, intervention_patterns):
        print(name, param.shape)

In [ ]:
from moe_reft import read_config
from moe_reft import interventions_config, tiny_sft
from moe_reft.olmoe import modeling_olmoe, configuration_olmoe, load_weights

config_path = "moe_reft/configs/olmoe.yaml"

train_config, interventions_config_, olmoe_config = read_config.load_all_configs(config_path)

custom_model = modeling_olmoe.OlmoeForCausalLM(
    configuration_olmoe.OlmoeInterventionsConfig(interventios_config=interventions_config_)
)



In [ ]:
import torch
from loguru import logger
# 2) Load HF weights into the overlapping parts, skipping interventions
report = load_weights.load_hf_into_custom_model(
    hf_model_name_or_path="allenai/OLMoE-1B-7B-0125-Instruct",
    custom_model=custom_model,
    intervention_patterns=["*.pre_moe_intervention.*", "*.after_moe_intervention.*"],
    map_dtype=torch.bfloat16,  # optional casting
    map_device=torch.device("cuda"),  # optional device move
    trust_remote_code=False,
)
logger.info(f"{report.summary()}")

for name, param in custom_model.named_parameters():
    if load_weights._matches_any(name, interventions_config.INTERVENTION_PATTERNS):
        param.requires_grad = True

# 5) Print parameter stats
total_params, trainable_params = load_weights._count_parameters(custom_model)
print(f"Total parameters:     {total_params}")
print(f"Trainable parameters: {trainable_params}")

logger.info(f"Parameter stats — total: {total_params}, trainable: {trainable_params}")
dataloader, _ = tiny_sft.build_tiny_sft_dataloader(model_name="allenai/OLMoE-1B-7B-0125-Instruct")

In [ ]:
from moe_reft import train

for step, batch in enumerate(dataloader):
    model_inputs, labels = train._unpack_batch(batch, torch.device("cuda" ))
    break

In [ ]:
from transformers import AutoTokenizer

prompt_template = [[{"role":"system","content":"<SYSTEMT>"},{"role":"user","content":"<USER>"},{"role":"assistant","content":"<ASSISTANT>"}]]

tokenizer = AutoTokenizer.from_pretrained("allenai/OLMoE-1B-7B-0125-Instruct")

In [15]:
chat_message = tokenizer.apply_chat_template(prompt_template, tokenize=True, return_dict=True)

In [16]:
chat_message

{'input_ids': [[50279, 29, 93, 10394, 49651, 187, 29, 47146, 53, 31, 187, 29, 93, 4537, 49651, 187, 29, 23131, 31, 187, 29, 93, 515, 5567, 49651, 187, 29, 1719, 5824, 1267, 5656, 31, 50279]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

In [9]:
enc = tokenizer(
            chat_message,
            add_special_tokens=False,
            return_tensors="pt"
        )

In [10]:
enc

{'input_ids': tensor([[50279,    29,    93, 10394, 49651,   187,    29, 47146,    53,    31,
           187,    29,    93,  4537, 49651,   187,    29, 23131,    31,   187,
            29,    93,   515,  5567, 49651,   187,    29,  1719,  5824,  1267,
          5656,    31, 50279]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [17]:
from moe_reft import sft_dataset

response_template = sft_dataset.extract_response_template(tokenizer)
rtmpl_ids = tokenizer.encode(response_template, return_tensors="pt").squeeze(0)

In [19]:
rtmpl_ids

tensor([  187,    29,    93,   515,  5567, 49651,   187])

In [21]:
import torch
from torch import Tensor

def find_after_subseq_batched(
    input_ids: Tensor,         # [B, T]
    response_ids: Tensor,      # [R]
) -> Tensor:                   # [B], indices after match, -1 if not found
    B, T = input_ids.shape
    R = response_ids.shape[0]

    if R > T:
        return input_ids.new_full((B,), -1, dtype=torch.long)

    # [B, T-R+1, R]
    windows = input_ids.unfold(dimension=1, size=R, step=1)

    # [B, T-R+1, R] == [1,1,R] -> [B, T-R+1, R]
    eq = windows == response_ids.view(1, 1, -1)
    matches = eq.all(dim=-1)  # [B, T-R+1]

    Lw = matches.shape[1]
    positions = torch.arange(Lw, device=input_ids.device)  # [T-R+1]
    pos = positions.unsqueeze(0).expand(B, -1)            # [B, T-R+1]

    # set non-matches to big sentinel
    sentinel = Lw + 1
    pos = pos.masked_fill(~matches, sentinel)

    first_pos, _ = pos.min(dim=1)  # [B]
    idx_after = torch.where(
        first_pos <= Lw,
        first_pos + R,
        input_ids.new_full((B,), -1, dtype=torch.long),
    )
    return idx_after


input_ids = torch.tensor([
    [1, 2, 3, 4, 5, 6],
    [3, 4, 1, 8, 7, 8],
])
response_ids = torch.tensor([3, 4])

idx_after = find_after_subseq_batched(input_ids, response_ids)
print(idx_after)  # tensor([4, 4])


tensor([4, 2])


In [27]:
idx_after[:,None]

tensor([[4],
        [2]])

In [28]:
label_ids = torch.arange(0,6)
labels = torch.where(label_ids[None,:]>=idx_after[:,None],input_ids, -100)

In [29]:
labels

tensor([[-100, -100, -100, -100,    5,    6],
        [-100, -100,    1,    8,    7,    8]])

In [ ]:
chat_message

In [ ]:
tokenizer.apply_chat_template(prompt_template[:-1], tokenizer=False, add_generation_prompt=True,tokenize=False)

In [ ]:
import re

def extract_user_segment(text: str) -> str | None:
    match = re.search(r"<USER>(.*?)<ASSISTANT>", text, flags=re.DOTALL)
    if not match:
        return None
    # Extract raw segment
    segment = match.group(1)
    # Remove any leading non-alphanumeric characters
    # Trim trailing spaces/newlines
    return segment


extract_user_segment(chat_message)

In [ ]:
tokenizer.apply_chat_template([{"role":"assistant","content":"<ASSISTANT>"}], tokenize=False)

In [ ]:
from typing import List

def find_subsequence(haystack: List[int], needle: List[int]) -> int:
    n, m = len(haystack), len(needle)
    for i in range(n - m + 1):
        if haystack[i : i + m] == needle:
            return i + m  # end index (1-based like your example)
    return -1

# Example
input_ids = [1, 2, 3,8,3,4, 5, 6]
response = [3, 4]

idx = find_subsequence(input_ids, response)
print(idx)  # 4


In [ ]:
input_ids[6:]

In [ ]:
tokenizer.encode("<ASSISTANT>") + [tokenizer.eos_token_id]

In [1]:
from moe_reft import sft_dataset

tokenizer_model = "allenai/OLMoE-1B-7B-0125-Instruct"
ds = sft_dataset.SFTDataset(
    source="openai/gsm8k",
    tokenizer_model_name=tokenizer_model,
    system_key=None,
    system_message="You are a helpful math tutor. Solve step by step.",
    user_key="question",
    assistant_key="answer",
    split="train",
    name="main",
)


/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2025-11-13 01:03:18.181 | INFO     | moe_reft.sft_dataset:__init__:49 - For the tokenizer_model_name='allenai/OLMoE-1B-7B-0125-Instruct' automatically assigned the response template to 
<|assistant|>

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
2025-11-13 01:03:20.479 | INFO     | moe_reft.sft_dataset:validate_one_sample:73 - Decoded Input (Full Prompt + Response)
|||IP_ADDRESS|||<|system|>
You are a helpful math tutor. Solve step by step.
<|user|>
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
<|assistant|>
Natalia sold 48/2 = <<48/2=24>>24 clips in 

In [2]:
for sample in ds:
    input_ids, labels = sample["input_ids"], sample["labels"]
    break
    

AssertionError: Length mismatch: system=155, user=155, assistant=126

In [6]:
from datasets import load_dataset

ds = load_dataset("openai/gsm8k",split="train",name="main")

In [7]:
ds[0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

In [1]:
from moe_reft import sft_dataset
from transformers import AutoTokenizer


tokenizer_model_name = "allenai/OLMoE-1B-7B-0125-Instruct"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_model_name)
train_dataset = sft_dataset.SFTDataset(
        source="openai/gsm8k",
        tokenizer=tokenizer,
        system_key=None,
        system_message="You are a helpful math tutor. Solve step by step.",
        user_key="question",
        assistant_key="answer",
        split="train",
        name="main",
    )
val_dataset = sft_dataset.SFTDataset(
    source="openai/gsm8k",
    tokenizer=tokenizer,
    system_key=None,
    system_message="You are a helpful math tutor. Solve step by step.",
    user_key="question",
    assistant_key="answer",
    split="test",
    name="main",
)

/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2025-11-16 19:15:29.880 | INFO     | moe_reft.sft_dataset:__init__:100 - For the tokenizer.name_or_path='allenai/OLMoE-1B-7B-0125-Instruct' automatically assigned the response template to 
<|assistant|>

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
2025-11-16 19:15:31.710 | INFO     | moe_reft.sft_dataset:validate_one_sample:124 - Decoded Input (Full Prompt + Response)
|||IP_ADDRESS|||<|system|>
You are a helpful math tutor. Solve step by step.
<|user|>
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
<|assistant|>
Natalia sold 48/2 = <<48/2=24>>24 clips

In [2]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Optional

import torch
from transformers import PreTrainedTokenizerBase

# Whatever you already use in SFTTransform
CROSS_ENTROPY_IGNORE_INDEX = -100  # or import from your constants


@dataclass
class SFTDataCollator:
    tokenizer: PreTrainedTokenizerBase
    label_pad_token_id: int = CROSS_ENTROPY_IGNORE_INDEX
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # 1. Separate labels so tokenizer.pad only sees model inputs
        labels_list: List[Any] = [f["labels"] for f in features]
        features_for_pad: List[Dict[str, Any]] = [
            {k: v for k, v in f.items() if k != "labels"} for f in features
        ]

        # 2. Let tokenizer.pad handle input_ids / attention_mask
        batch = self.tokenizer.pad(
            features_for_pad,
            padding=True,                 # pad to max length in this batch
            max_length=None,              # or a fixed max_length if you want
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        # 3. Manually pad labels to match seq_len of input_ids
        seq_len: int = batch["input_ids"].size(1)
        padded_labels: List[List[int]] = []

        for lbl in labels_list:
            # convert to python list of ints
            if isinstance(lbl, torch.Tensor):
                lbl_list = lbl.tolist()
            else:
                lbl_list = list(lbl)

            # truncate if somehow longer than seq_len
            if len(lbl_list) > seq_len:
                lbl_list = lbl_list[:seq_len]

            pad_len = seq_len - len(lbl_list)
            if pad_len > 0:
                lbl_list = lbl_list + [self.label_pad_token_id] * pad_len

            padded_labels.append(lbl_list)

        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)

        return batch


In [3]:
from torch.utils.data import DataLoader

collator = SFTDataCollator(tokenizer=train_dataset.tokenizer)

train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=1,
        pin_memory=True,
        collate_fn = collator
    )

In [10]:
# for td in train_loader:
#     print(td['input_ids'].shape)
    # break

In [5]:
td['input_ids'].shape

torch.Size([32, 376])

In [1]:
from transformers import AutoTokenizer
from moe_reft import sft_dataset

tokenizer_model_name = "allenai/OLMoE-1B-7B-0125-Instruct"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_model_name)
response_template = sft_dataset.extract_response_template(tokenizer)


/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [4]:
tokenizer(response_template)['input_ids']

[187, 29, 93, 515, 5567, 49651, 187]